<a href="https://colab.research.google.com/github/ckrickyh/pythonTools/blob/main/gestureRecongization_mediapipe_img.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# https://blog.csdn.net/qq_42535394/article/details/148613545
# https://mediapipe.readthedocs.io/en/latest/solutions/hands.html
# https://medium.com/@ohr.morris/leveraging-google-mediapipe-for-easy-hand-gesture-classification-8ff2365c202c
import cv2
import mediapipe as mp
import math
from pathlib import Path
import pandas as pd
import numpy as np

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

# 根據兩點的座標，計算角度
def vector_2d_angle(v1, v2):
    v1_x = v1[0]
    v1_y = v1[1]
    v2_x = v2[0]
    v2_y = v2[1]
    try:
        angle_= math.degrees(math.acos((v1_x*v2_x+v1_y*v2_y)/(((v1_x**2+v1_y**2)**0.5)*((v2_x**2+v2_y**2)**0.5))))
    except:
        angle_ = 180
    return angle_

# 根據傳入的 21 個節點座標，得到該手指的角度
def hand_angle(hand_):
    angle_list = []
    # thumb 大拇指角度
    angle_ = vector_2d_angle(
        ((int(hand_[0][0])- int(hand_[2][0])),(int(hand_[0][1])-int(hand_[2][1]))),
        ((int(hand_[3][0])- int(hand_[4][0])),(int(hand_[3][1])- int(hand_[4][1])))
        )
    angle_list.append(angle_)
    # index 食指角度
    angle_ = vector_2d_angle(
        ((int(hand_[0][0])-int(hand_[6][0])),(int(hand_[0][1])- int(hand_[6][1]))),
        ((int(hand_[7][0])- int(hand_[8][0])),(int(hand_[7][1])- int(hand_[8][1])))
        )
    angle_list.append(angle_)
    # middle 中指角度
    angle_ = vector_2d_angle(
        ((int(hand_[0][0])- int(hand_[10][0])),(int(hand_[0][1])- int(hand_[10][1]))),
        ((int(hand_[11][0])- int(hand_[12][0])),(int(hand_[11][1])- int(hand_[12][1])))
        )
    angle_list.append(angle_)
    # ring 無名指角度
    angle_ = vector_2d_angle(
        ((int(hand_[0][0])- int(hand_[14][0])),(int(hand_[0][1])- int(hand_[14][1]))),
        ((int(hand_[15][0])- int(hand_[16][0])),(int(hand_[15][1])- int(hand_[16][1])))
        )
    angle_list.append(angle_)
    # pink 小拇指角度
    angle_ = vector_2d_angle(
        ((int(hand_[0][0])- int(hand_[18][0])),(int(hand_[0][1])- int(hand_[18][1]))),
        ((int(hand_[19][0])- int(hand_[20][0])),(int(hand_[19][1])- int(hand_[20][1])))
        )
    angle_list.append(angle_)
    return angle_list

# 根據手指角度的串列內容，返回對應的手勢名稱
def hand_pos(finger_angle):
    f1 = finger_angle[0]   # 大拇指角度
    f2 = finger_angle[1]   # 食指角度
    f3 = finger_angle[2]   # 中指角度
    f4 = finger_angle[3]   # 無名指角度
    f5 = finger_angle[4]   # 小拇指角度

    # 小於 50 表示手指伸直，大於等於 50 表示手指捲縮
    if f1<50 and f2>=50 and f3>=50 and f4>=50 and f5>=50:
        return 'good'
    elif f1>=50 and f2>=50 and f3<50 and f4>=50 and f5>=50:
        return 'no!!!'
    elif f1<50 and f2<50 and f3>=50 and f4>=50 and f5<50:
        return 'ROCK!'
    elif f1>=50 and f2>=50 and f3>=50 and f4>=50 and f5>=50:
        return '0'
    elif f1>=50 and f2>=50 and f3>=50 and f4>=50 and f5<50:
        return 'pink'
    elif f1>=50 and f2<50 and f3>=50 and f4>=50 and f5>=50:
        return '1'
    elif f1>=50 and f2<50 and f3<50 and f4>=50 and f5>=50:
        return '2'
    elif f1>=50 and f2>=50 and f3<50 and f4<50 and f5<50:
        return 'ok'
    elif f1<50 and f2>=50 and f3<50 and f4<50 and f5<50:
        return 'ok'
    elif f1>=50 and f2<50 and f3<50 and f4<50 and f5>50:
        return '3'
    elif f1>=50 and f2<50 and f3<50 and f4<50 and f5<50:
        return '4'
    elif f1<50 and f2<50 and f3<50 and f4<50 and f5<50:
        return '5'
    elif f1<50 and f2>=50 and f3>=50 and f4>=50 and f5<50:
        return '6'
    elif f1<50 and f2<50 and f3>=50 and f4>=50 and f5>=50:
        return '7'
    elif f1<50 and f2<50 and f3<50 and f4>=50 and f5>=50:
        return '8'
    elif f1<50 and f2<50 and f3<50 and f4<50 and f5>=50:
        return '9'
    else:
        return ''

# file = cv2.imread("/Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/empty.jpeg")

dics = []

IMAGE_FILES = []
folderPath = r'/Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo'

for file in Path(folderPath).iterdir():
    if file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        IMAGE_FILES.append(file)

with mp_hands.Hands(
    static_image_mode = True,
    max_num_hands=2,
    min_detection_confidence=0.5) as hands:

   for idx, file in enumerate(IMAGE_FILES):
        img = cv2.flip(cv2.imread(file), 1)
        img2 = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(img2)

        w, h = 540, 310

        #print(f'Handedness: {results.multi_hand_landmarks}')
        if not results.multi_hand_landmarks:
            print(f'filePath: {file}, index: {idx}, text: NA')
            dic = {'file': file, 'index': idx, 'number': 'NA'}
            dics.append(dic)
            continue

        for hand_landmarks in results.multi_hand_landmarks:
            finger_points = []                   # 記錄手指節點座標的串列
            for i in hand_landmarks.landmark:
                # 將 21 個節點換算成座標，記錄到 finger_points
                x = i.x*w
                y = i.y*h
                finger_points.append((x,y))
                #print(f'fp: {finger_points}')
            if finger_points:
                finger_angle = hand_angle(finger_points) # 計算手指角度，回傳長度為 5 的串列
                #print(finger_angle)                     # 印出角度 ( 有需要就開啟註解 )
                text = hand_pos(finger_angle)            # 取得手勢所回傳的內容
                print(f'filePath: {file}, index: {idx}, text: {text}')
                dic = {'file': file, 'index': idx, 'number': text}
                dics.append(dic)

df = pd.DataFrame(dics)

I0000 00:00:1755486547.728240 4727255 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
W0000 00:00:1755486547.739311 4833533 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1755486547.748510 4833535 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


filePath: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/WhatsApp Image 2025-08-17 at 23.42.13 (1).jpeg, index: 0, text: NA
filePath: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/finger6 copy 2.jpeg, index: 1, text: 6
filePath: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/WhatsApp Image 2025-08-17 at 23.42.13 (2) copy.jpeg, index: 2, text: 4
filePath: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/WhatsApp Image 2025-08-17 at 23.42.13 (2).jpeg, index: 3, text: 4
filePath: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/finger6 copy 3.jpeg, index: 4, text: 6
filePath: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/zempty.jpeg, index: 5, text: NA
filePath: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/finger6.jpeg, index: 6, text: 6
filePath: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/WhatsApp Image 2025-08-17 at 2

In [ ]:
#"Convert the continuous gesture-numbered photos into a string of numbers and bring it to the new column of the first photo."
for index, row in df.iterrows():
  if row['number'] != 'NA':
    thisNumber = str(row['number'])
    i = 1
    combine_num = thisNumber

    if index - 1 > 0:
      if df.iloc[index -1]['number'] != 'NA':
        df.iloc[index]['number'] = np.nan

    while True:
      if (index + i) < df.index.size:
        next_number = str(df.iloc[index + i]['number'])
        if next_number != 'NA':
          combine_num = combine_num + str(next_number)
          i = i + 1
        else:
          if df.at[index - 1, 'number'] != 'NA':
            df.at[index, 'itemNum'] = np.nan
            break
          else:
            df.at[index, 'itemNum'] = np.int64(combine_num)
          break

      else:
        df.at[index, 'itemNum'] = np.int64(thisNumber)
        break

df

/var/folders/3x/v2v5lgj97jqg15bcfj_8xq5m0000gn/T/ipykernel_72359/970977913.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[index]['number'] = np.nan


,file,index,number,itemNum
0,/Users/hochakkong/Documents/python3.9_venv/pro...,0,NA,NaN
1,/Users/hochakkong/Documents/python3.9_venv/pro...,1,6,6446.0
2,/Users/hochakkong/Documents/python3.9_venv/pro...,2,4,NaN
3,/Users/hochakkong/Documents/python3.9_venv/pro...,3,4,NaN
4,/Users/hochakkong/Documents/python3.9_venv/pro...,4,6,NaN
5,/Users/hochakkong/Documents/python3.9_venv/pro...,5,NA,NaN
6,/Users/hochakkong/Documents/python3.9_venv/pro...,6,6,60.0
7,/Users/hochakkong/Documents/python3.9_venv/pro...,7,0,NaN
8,/Users/hochakkong/Documents/python3.9_venv/pro...,8,NA,NaN
9,/Users/hochakkong/Documents/python3.9_venv/pro...,9,4,4.0


In [ ]:
# the photo wihtout number-recongizied will be assigned the previous photo with number-reconginzied

conditions = [df['itemNum'].notna(), df['itemNum'].isna()]
choices = ['identifer', df['itemNum']]
df['identifer'] = np.select(conditions, choices, default = np.nan)
df['classifer'] = df['itemNum'].fillna(method='ffill')
# df['cumCount'] = df.groupby(['number','classifer']).cumcount()+1
df['cumCount'] = df[(df['number']=='NA')&(df['classifer'].notna())].groupby(['number', 'classifer']).cumcount()+1
df['cumCount'] = df['cumCount'].fillna(0).astype('int')

conditions = [df['classifer'].isna() , df['identifer'] == 'identifer', df['cumCount']==0, df['identifer']!='identifer']
choices = ['TreePhoto', 'identifer', 'identifer', df['cumCount']]
df['cumCount'] = np.select(conditions, choices, default = np.nan)

df['cumCount'] = df['cumCount'].replace('nan', 'identifer')

df['TreeNo'] = df['classifer'].fillna(0).astype(int)
df['TreeNoText'] = df['TreeNo'].apply(lambda: x, str(x).zfill(len(str(df['TreeNO'].max()))))

conditions2 = [df['cumCount'].isin(['identifer','TreePhoto']), ~df['cumCount'].isin(['identifer','TreePhoto'])]
choices2 = ['T' + df['TreeNoText'] + '_identifer' + df['index'].astype(str), 'T'+df['TreeNoText'] + '_' + df['cumCount'].astype(str)]
df['photoName'] = np.select(conditions2, choices2, default = np.nan)

df

/var/folders/3x/v2v5lgj97jqg15bcfj_8xq5m0000gn/T/ipykernel_72359/600276823.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['classifer'] = df['itemNum'].fillna(method='ffill')


,file,index,number,itemNum,identifer,classifer,cumCount,TreeNo,photoName
0,/Users/hochakkong/Documents/python3.9_venv/pro...,0,NA,NaN,nan,NaN,TreePhoto,0,T0_identifer0
1,/Users/hochakkong/Documents/python3.9_venv/pro...,1,6,6446.0,identifer,6446.0,identifer,6446,T6446_identifer1
2,/Users/hochakkong/Documents/python3.9_venv/pro...,2,4,NaN,nan,6446.0,identifer,6446,T6446_identifer2
3,/Users/hochakkong/Documents/python3.9_venv/pro...,3,4,NaN,nan,6446.0,identifer,6446,T6446_identifer3
4,/Users/hochakkong/Documents/python3.9_venv/pro...,4,6,NaN,nan,6446.0,identifer,6446,T6446_identifer4
5,/Users/hochakkong/Documents/python3.9_venv/pro...,5,NA,NaN,nan,6446.0,1,6446,T6446_1
6,/Users/hochakkong/Documents/python3.9_venv/pro...,6,6,60.0,identifer,60.0,identifer,60,T60_identifer6
7,/Users/hochakkong/Documents/python3.9_venv/pro...,7,0,NaN,nan,60.0,identifer,60,T60_identifer7
8,/Users/hochakkong/Documents/python3.9_venv/pro...,8,NA,NaN,nan,60.0,1,60,T60_1
9,/Users/hochakkong/Documents/python3.9_venv/pro...,9,4,4.0,identifer,4.0,identifer,4,T4_identifer9


# File Rename

In [ ]:
# rename file

for index, row in df.iterrows():
  original_path = Path(row['file'])
  new_file_name = str((row['photoName']))+'.jpg'

    # Construct the new file path
  new_path = original_path.parent/new_file_name

  # Rename the file
  try:
      original_path.rename(new_path)
      print(f'Renamed: {original_path} to {new_path}')
  except FileNotFoundError:
      print(f'File not found: {original_path}')
  except Exception as e:
      print(f'Error renaming {original_path}: {e}')

Renamed: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/WhatsApp Image 2025-08-17 at 23.42.13 (1).jpeg to /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/T0_identifer0.jpg
Renamed: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/finger6 copy 2.jpeg to /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/T6446_identifer1.jpg
Renamed: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/WhatsApp Image 2025-08-17 at 23.42.13 (2) copy.jpeg to /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/T6446_identifer2.jpg
Renamed: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/WhatsApp Image 2025-08-17 at 23.42.13 (2).jpeg to /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/T6446_identifer3.jpg
Renamed: /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe/photo/finger6 copy 3.jpeg to /Users/hochakkong/Documents/python3.9_venv/projects/mediapipe

In [ ]:
str(20).zfill(len(str(20))+1)

'020'